In [1]:
import sys
sys.path.append("/project/src")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import uniform, loguniform

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.model_selection import validation_curve
from sklearn.pipeline import Pipeline
from lifelines import CoxPHFitter

from coxmodel import CoxPH
from preprocessing import (
    split_features_target,
    concat_features_target,
    SURVIVAL_EVENT_COL,
    SURVIVAL_TIME_COL,
    BaseDatasetPreprocessor,
)
from helpers import save_pic

import wandb
import joblib

# Read datasets

In [3]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive # type: ignore
    drive.mount('/content/drive')
    train_df_csv = "/content/drive/MyDrive/bachelor/nacc_train_reduced.csv"
else:
    train_df_csv = "./data/nacc_train_reduced.csv"

In [4]:
train_df = pd.read_csv(train_df_csv, delimiter=',')

In [5]:
train_df.shape

(6499, 141)

## Add time-interaction terms

The Schoenfeld residuals test showed that `ANIMALS` and `HRATE` have borderline violations of the proportional hazard assumption. The functional form is correct (confirmed by Martingale residuals), so the issue is a time-varying effect.

To address this, we add interaction terms `ANIMALS × TIME` and `HRATE × TIME`. This allows the Cox model to capture how the effect of these variables changes over time, satisfying the proportional hazard assumption.

In [6]:
term_interactions_cols = ['ANIMALS', 'HRATE']

In [ ]:
for col in term_interactions_cols:
    if col in train_df.columns:
        train_df[f'{col}_interaction'] = train_df[col] * train_df[SURVIVAL_TIME_COL]
        print(f"Added {col}_interaction column.")
    else:
        term_interactions_cols.remove(col)
        print(f"{col} column not found in train_df — skipping interaction term")

Added ANIMALS_interaction column.
Added HRATE_interaction column.
Shape after interaction terms: (6499, 143)


## Compute X and y and calculate weights

In [ ]:
train_X, train_y = split_features_target(train_df)

# Initialize wandb

In [ ]:
features_num = train_df.shape[1]
n_samples = train_df.shape[0]

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="semariik",
    # Set the wandb project where this run will be logged.
    project="survival-analysis-mci",
)

wandb.define_metric("cox/*", step_metric="cox/lambda")

# Parameters and constents definition

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
duration_col = "TIME"
event_col    = "EVENT_MCI"
lambdas      = np.logspace(-1, 2, 50)

In [ ]:
param_distributions = {
    'model__penalizer': loguniform(0.1, 100),
    'model__l1_ratio':  uniform(0, 1),
}

In [ ]:
cox_tunning_pipeline = Pipeline([
  ('preprocessor', BaseDatasetPreprocessor()),
  ("model", CoxPH(duration_col=SURVIVAL_TIME_COL, event_col=SURVIVAL_EVENT_COL, compute_weights=True))
])

# Cox model tunning

## Tune parameters with GridSearchCV

In [ ]:
def cox_concordance_scorer(estimator, X, y):
    # with warnings.catch_warnings(record=True) as caught:
    #     warnings.simplefilter("always", ConvergenceWarning)
    #     score = estimator.score(X, y)

    # for w in caught:
    #     msg = str(w.message)
    #     if "Newton-Raphson failed" in msg or "suspiciously close to 0" in msg:
    #         return 0.5

    score = estimator.score(X, y)

    return score

In [ ]:
cv_splits = list(cv.split(train_X, train_y[event_col].astype(int)))

random_search = RandomizedSearchCV(
    estimator=cox_tunning_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    cv=cv_splits,
    scoring=cox_concordance_scorer,
    refit=True,
    error_score=0.5
)

In [ ]:
random_search.fit(train_X, train_y)

## Asses overfitting and underfitting and select optimal lambda

Create error curves for cross validation and train. Check that optimally selected parameters are the optimal point,w here no overfit no undefit. Reselect better lambda, in case of need by graph

In [ ]:
# train_errors = []
# val_errors   = []
# val_stds     = []

# for penalizer in lambdas:
#     fold_train = []
#     fold_val   = []

#     for train_idx, val_idx in cv_splits:
#         X_train_fold = train_X.iloc[train_idx]
#         y_train_fold = train_y[train_idx]
#         X_val_fold   = train_X.iloc[val_idx]
#         y_val_fold   = train_y[val_idx]

#         model = CoxPH(penalizer=penalizer, l1_ratio=0.0, duration_col=duration_col, event_col=event_col, compute_weights=True)
#         model.fit(X_train_fold, y_train_fold)

#         c_index_train = model.score(X_train_fold, y_train_fold)
#         c_index_val   = model.score(X_val_fold,   y_val_fold)

#         fold_train.append(1 - c_index_train)
#         fold_val.append(1 - c_index_val)

#     train_errors.append(np.mean(fold_train))
#     val_errors.append(np.mean(fold_val))
#     val_stds.append(np.std(fold_val) / np.sqrt(5))

#     wandb.log({
#         "cox/lambda":      penalizer,
#         "cox/log_lambda":  np.log(penalizer),
#         "cox/train_error": train_errors[-1],
#         "cox/val_error":   val_errors[-1],
#         "cox/gap":         val_errors[-1] - train_errors[-1],
#     })

In [ ]:
random_search.best_params_

In [ ]:
best_model_fit_assesment_pipeline = Pipeline([
  ('preprocessor', BaseDatasetPreprocessor()),
  ("model", CoxPH(duration_col=SURVIVAL_TIME_COL, event_col=SURVIVAL_EVENT_COL, compute_weights=True, penalizer=random_search.best_params_['model__penalizer'], l1_ratio=random_search.best_params_['model__l1_ratio']))
])

In [ ]:
train_scores, val_scores = validation_curve(
    estimator=best_model_fit_assesment_pipeline,
    X=train_X,
    y=train_y,
    param_name='model__penalizer',
    param_range=lambdas,
    cv=cv_splits,
    scoring=cox_concordance_scorer,
    error_score=0.5,
)

train_errors = 1 - np.mean(train_scores, axis=1)
val_errors   = 1 - np.mean(val_scores,   axis=1)
val_stds     = np.std(val_scores, axis=1) / np.sqrt(3)

In [ ]:
for i, penalizer in enumerate(lambdas):
    wandb.log({
        "cox/lambda":      penalizer,
        "cox/train_error": train_errors[i],
        "cox/val_error":   val_errors[i],
        "cox/gap":         val_errors[i] - train_errors[i],
    })

In [ ]:
lambdas

In [ ]:
train_errors

In [ ]:
val_errors

In [ ]:
best_penalizer = random_search.best_params_['model__penalizer']

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lambdas, train_errors, color="blue", linewidth=2, label="Train error")
ax.plot(lambdas, val_errors,   color="red",  linewidth=2, label="Validation error (CV)")

ax.fill_between(lambdas, val_errors - val_stds, val_errors + val_stds,
                alpha=0.2, color="red", label="Val ± std")
ax.fill_between(lambdas, train_errors, val_errors,
                alpha=0.15, color="orange", label="Gap (overfitting signal)")

ax.axvline(x=best_penalizer, color="green", linestyle="--", linewidth=1.5,
           label=f"Best penalizer ({best_penalizer:.4f})")

ax.set_xscale("log")
ax.set_xlabel("lambda", fontsize=12)
ax.set_ylabel("Error", fontsize=12)
ax.set_title("Train vs Validation Error — Regularization Diagnostic (Cox)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_pic(plt, "cox_lambda_curves.png")
plt.show()

In [ ]:
cox_best_params = random_search.best_params_
cox_best_params

In [ ]:
cv_cindex = random_search.best_score_
cv_cindex

In [ ]:
cv_cindex_std = random_search.cv_results_['std_test_score'][random_search.best_index_] 
cv_cindex_std

In [ ]:
wandb.log({"best_parameters": cox_best_params})
joblib.dump(cox_best_params, "joblib-storage/cox_best_params.joblib")
joblib.dump(random_search.best_estimator_, "joblib-storage/cox_best_pipeline.joblib")
joblib.dump(cv_cindex, "joblib-storage/cox_best_cv_cindex.joblib")
joblib.dump(cv_cindex_std, "joblib-storage/cox_best_cv_cindex_std.joblib")
wandb.finish()

# Assert hazard assumptions on final dataset

In [ ]:
preprocessor = random_search.best_estimator_.named_steps['preprocessor']
preprocessed_train_X = preprocessor.transform(train_X)

preprocessed_train_df = concat_features_target(preprocessed_train_X, train_y)

cph = CoxPHFitter(penalizer=0.4, l1_ratio=0.027)
cph.fit(preprocessed_train_df, duration_col=SURVIVAL_TIME_COL, event_col=SURVIVAL_EVENT_COL)
cph.check_assumptions(preprocessed_train_df, p_value_threshold=0.05, show_plots=True, columns=term_interactions_cols)